# Experiment: AutoEIT Task II Scoring

"
"Objective:
"
"- Implement a reproducible, deterministic script for **Specific Test II** (meaning-based EIT scoring).
"
"- Generate sentence-level scores for all utterances and write scored Excel/CSV outputs.

"
"Success criteria:
"
"- Runs end-to-end from source workbook to scored outputs.
"
"- Uses explicit rubric rules (0-4) with stable behavior.
"
"- Reports agreement metrics against human ratings where available.


## Dataset Loading and Runtime Configuration

"
"This notebook reads the provided workbook, executes the scorer from `autoeit_scorer.py`, and writes:
"
"- `AutoEIT_Sample_Transcriptions_Scored.xlsx`
"
"- `AutoEIT_scores.csv`


In [1]:
from pathlib import Path
import pandas as pd

from autoeit_scorer import (
    run_scoring_pipeline,
    compute_metrics,
    write_output_workbook,
    score_utterance,
)

repo_dir = Path.cwd()
source = repo_dir / "Example_EIT Transcription and Scoring Sheet.xlsx"
output_xlsx = repo_dir / "AutoEIT_Sample_Transcriptions_Scored.xlsx"
output_csv = repo_dir / "AutoEIT_scores.csv"

assert source.exists(), f"Missing input workbook: {source}"
source


PosixPath('/Users/fallofpheonix/Project/Human AI/AutoEIT-STS/Example_EIT Transcription and Scoring Sheet.xlsx')

## Rubric Implementation

"
"Scoring is implemented in `score_utterance(target_raw, learner_raw)` as a deterministic rule cascade:
"
"- **4**: exact/near exact meaning-preserving reproduction after normalization
"
"- **3**: near-accurate with minor grammatical/morphological deviation
"
"- **2**: partial meaning preserved
"
"- **1**: minimal fragmentary meaning
"
"- **0**: no meaning preserved / wrong language / no response

"
"Normalization includes stripping stimulus word-count hints `(N)`, removing bracket annotations, punctuation/diacritics normalization, and whitespace collapsing.


In [2]:
rubric_examples = [
    ("Quiero cortarme el pelo (7)", "Quiero cortarme el pelo"),
    ("Puede que llueva mañana (8)", "[puede que llue-] Puede que llueva mañana"),
    ("Ellos están en casa (5)", "Ellos son en casa"),
    ("Necesito comprar pan y leche (6)", "comprar pan"),
    ("Me gusta estudiar español (5)", "[en inglés] I like studying Spanish"),
]

rows = []
for tgt, hyp in rubric_examples:
    s, r = score_utterance(tgt, hyp)
    rows.append({"target": tgt, "learner": hyp, "auto_score": s, "rationale": r})

pd.DataFrame(rows)


,target,learner,auto_score,rationale
0,Quiero cortarme el pelo (7),Quiero cortarme el pelo,4,Exact reproduction after normalisation
1,Puede que llueva mañana (8),[puede que llue-] Puede que llueva mañana,4,Exact reproduction after normalisation
2,Ellos están en casa (5),Ellos son en casa,4,"1-word deviation, full meaning preserved (edit..."
3,Necesito comprar pan y leche (6),comprar pan,2,"Partial meaning (content_overlap=0.50, edit=3,..."
4,Me gusta estudiar español (5),[en inglés] I like studying Spanish,0,Response in wrong language [en inglés]


## End-to-End Scoring Run

In [3]:
df = run_scoring_pipeline(str(source))

preview_cols = [
    "participant_id", "version", "sentence_id",
    "human_score", "auto_score", "score_diff", "rationale"
]
df[df["has_human_score"]][preview_cols].head(30)


Loading dataset from: /Users/fallofpheonix/Project/Human AI/AutoEIT-STS/Example_EIT Transcription and Scoring Sheet.xlsx


  → 1740 utterances from 29 participants


,participant_id,version,sentence_id,human_score,auto_score,score_diff,rationale
180,4,vA,1,4.0,4,0.0,Exact reproduction after normalisation
181,4,vA,2,4.0,4,0.0,Exact reproduction after normalisation
182,4,vA,3,4.0,4,0.0,Exact reproduction after normalisation
183,4,vA,4,4.0,4,0.0,Exact reproduction after normalisation
184,4,vA,5,4.0,4,0.0,Exact reproduction after normalisation
185,4,vA,6,4.0,4,0.0,Exact reproduction after normalisation
186,4,vA,7,4.0,4,0.0,Exact reproduction after normalisation
187,4,vA,8,4.0,4,0.0,Exact reproduction after normalisation
188,4,vA,9,4.0,4,0.0,Exact reproduction after normalisation
189,4,vA,10,4.0,4,0.0,Exact reproduction after normalisation


## Evaluation Explanation

"
"Evaluation compares automated sentence-level scores against available human scores:
"
"- Exact agreement rate
"
"- Within-1 agreement rate
"
"- Participant-level total score deviation
"
"- Confusion matrix (human vs automated score classes)


In [4]:
metrics = compute_metrics(df)

for k, v in metrics.items():
    if k != "confusion_summary":
        print(f"{k}: {v}")

print("\nconfusion_summary:")
metrics["confusion_summary"]


n_rated_utterances: 1560
exact_agreement_rate_pct: 74.04
within1_agreement_pct: 96.22
mean_participant_deviation: 7.31
max_participant_deviation: 23
pct_participants_within10: 76.92
n_participants: 52
auto_score_distribution: {0: 82, 1: 11, 2: 147, 3: 125, 4: 1195}
human_score_distribution: {0.0: 92, 1.0: 48, 2.0: 177, 3.0: 290, 4.0: 953}

confusion_summary:


Auto,0,1,2,3,4
Human,,,,,
0.0,77,5,10,0,0
1.0,5,4,37,2,0
2.0,0,2,69,59,47
3.0,0,0,31,58,201
4.0,0,0,0,6,947


## Output Generation

"
"The following cell writes scored outputs in the expected submission formats.


In [5]:
write_output_workbook(df, str(source), str(output_xlsx))

df[[
    "participant_id", "version", "sentence_id", "stimulus",
    "transcription", "human_score", "auto_score", "rationale"
]].to_csv(output_csv, index=False)

print(f"Excel: {output_xlsx}")
print(f"CSV:   {output_csv}")


  → Output saved: /Users/fallofpheonix/Project/Human AI/AutoEIT-STS/AutoEIT_Sample_Transcriptions_Scored.xlsx
Excel: /Users/fallofpheonix/Project/Human AI/AutoEIT-STS/AutoEIT_Sample_Transcriptions_Scored.xlsx
CSV:   /Users/fallofpheonix/Project/Human AI/AutoEIT-STS/AutoEIT_scores.csv


## Notes

"
"- Runtime complexity per utterance is dominated by token-level Levenshtein distance: **O(m*n)**.
"
"- For dataset size `U` with average length `L`, total runtime is approximately **O(U*L^2)**.
"
"- The scorer is deterministic: no stochastic model components are used.
